# Sycophancy in LLM-Based MT Evaluation — Analysis Notebook

Generates all 5 figures + stats tables for the paper. Run cells top-to-bottom after `python -m src.runner --config configs/full.yaml` has populated `results/results.jsonl`.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys; sys.path.insert(0, '..')
from analysis.stats import to_long, add_metrics, h1_wilcoxon, h1_directional_rates, h3_ambiguity_moderation, h4_prompt_resistance, judge_human_correlation, CHALLENGE_KINDS

FIGS = Path('../figs'); FIGS.mkdir(exist_ok=True)
sns.set_context('paper'); sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300

In [ ]:
rows = [json.loads(l) for l in open('../results/results.jsonl')]
wide = pd.DataFrame(rows)
long = to_long(wide)
long = add_metrics(long)
print(wide.shape, long.shape)
long.head()

## H1: sycophancy exists (Wilcoxon + compliance rates)

In [ ]:
h1 = h1_wilcoxon(long); display(h1)
rates = h1_directional_rates(long); display(rates)

## Figure 1: score shift vs ambiguity decile

In [ ]:
challenged = long[long['condition'].isin(CHALLENGE_KINDS)].copy()
challenged['ambig_decile'] = pd.qcut(challenged['ambiguity'], q=10, labels=False, duplicates='drop')
agg = challenged.groupby(['ambig_decile', 'judge'])['shift'].apply(lambda x: x.abs().mean()).reset_index()
fig, ax = plt.subplots(figsize=(6,3.5))
for j, sub in agg.groupby('judge'):
    ax.plot(sub['ambig_decile'], sub['shift'], marker='o', label=j)
ax.set_xlabel('Ambiguity decile (low → high)'); ax.set_ylabel('Mean |score shift| after challenge')
ax.legend(); fig.tight_layout(); fig.savefig(FIGS/'fig1_shift_vs_ambiguity.png')

## Figure 2: regressive sycophancy rate by challenge type

In [ ]:
fig, ax = plt.subplots(figsize=(6,3.5))
sns.barplot(data=rates, x='condition', y='regressive_rate', hue='judge', ax=ax)
ax.set_ylabel('Regressive sycophancy rate'); ax.set_xlabel('Condition')
fig.tight_layout(); fig.savefig(FIGS/'fig2_regressive_by_challenge.png')

## Figure 3: heatmap lang_pair × ambiguity tertile

In [ ]:
heat = challenged.groupby(['lang_pair', 'ambiguity_tertile'])['compliant'].mean().unstack()
fig, ax = plt.subplots(figsize=(5,3))
sns.heatmap(heat, annot=True, fmt='.2f', cmap='Reds', ax=ax)
ax.set_title('Compliance rate by language pair × ambiguity tertile')
fig.tight_layout(); fig.savefig(FIGS/'fig3_heatmap_pair_x_tier.png')

## Table 1: judge-human Spearman, baseline vs challenged vs controls

In [ ]:
table1 = judge_human_correlation(long); display(table1)
table1.to_csv('../paper/figures/table1_correlations.csv', index=False)

## H3 ambiguity moderation + H4 prompt resistance

In [ ]:
display(h3_ambiguity_moderation(long))
display(h4_prompt_resistance(long))

## Stretch: Mixed-effects regression

In [ ]:
from analysis.mem import fit_mem, coef_table
result = fit_mem(long)
display(coef_table(result))